# Rol 1 – Limpieza e integración de las bases públicas
**Proyecto 2 DATA-1002:** turismo y conciertos/festivales en Medellín.

Este notebook lee las bases originales de `data/raw` (sin modificarlas), las limpia y las une en una tabla mensual:
`data/processed/panel_publico.csv`.

**Bases usadas:** (1) Tráfico vehicular ANI, (2) DANE EMA (ocupación e índice de tarifas, Antioquia), (3) Migración Colombia (entradas de extranjeros por el aeropuerto José María Córdova).

**Periodo:** 2019-01 a 2025-12 (84 meses).

In [1]:
import pandas as pd

RAW = "../data/raw/"
PROC = "../data/processed/"

## 1. Tráfico vehicular ANI
Esta base trae el tráfico de **todos los peajes del país**, por categoría de vehículo y por mes.

In [2]:
ani = pd.read_csv(RAW + "trafico_ani.csv", dtype=str)
print(ani.shape)
ani.head()

(149944, 9)


,IdPeaje,Peaje,CategoriaTarifa,Desde,Hasta,ValorTarifa,CantidadTrafico,CantidadEvasores,CantidadExentos787
0,1,ALVARADO,I,2015 Aug 01 12:00:00 AM,2015 Aug 31 12:00:00 AM,"7,000","27,730",0,660
1,1,ALVARADO,II,2015 Aug 01 12:00:00 AM,2015 Aug 31 12:00:00 AM,"7,600","9,930",0,70
2,1,ALVARADO,III,2015 Aug 01 12:00:00 AM,2015 Aug 31 12:00:00 AM,"16,100","1,277",0,0
3,1,ALVARADO,IV,2015 Aug 01 12:00:00 AM,2015 Aug 31 12:00:00 AM,"20,400","2,074",0,0
4,1,ALVARADO,V,2015 Aug 01 12:00:00 AM,2015 Aug 31 12:00:00 AM,"22,900","4,167",0,0


**Elegir los peajes.** Nos quedamos con **Las Palmas** y **Guarne**: están en la ruta Medellín – Oriente antioqueño / aeropuerto José María Córdova (criterio nuestro, por confirmar) y tienen la serie completa (2014-01 a 2026-05). Primero revisamos que existan y cuántos meses tienen:

In [3]:
peajes = ["LAS PALMAS", "GUARNE"]
ani = ani[ani["Peaje"].isin(peajes)].copy()
print(ani.shape)
ani["Peaje"].value_counts()

(2910, 9)


Peaje
LAS PALMAS    1609
GUARNE        1301
Name: count, dtype: int64

**Fechas y números.** La fecha viene como texto en inglés (`2015 Aug 01 12:00:00 AM`) y las cantidades como texto con comas (`27,730`). Los convertimos.

In [4]:
ani["mes"] = pd.to_datetime(ani["Desde"], format="%Y %b %d %I:%M:%S %p")
ani["trafico"] = pd.to_numeric(ani["CantidadTrafico"].str.replace(",", ""), errors="coerce")

print("Faltantes en tráfico:", ani["trafico"].isna().sum())
print("Duplicados (peaje, categoría, mes):", ani.duplicated(["IdPeaje", "CategoriaTarifa", "mes"]).sum())

Faltantes en tráfico: 0
Duplicados (peaje, categoría, mes): 0


**Agregar por mes.** Sumamos todas las categorías para tener el tráfico total por peaje y mes, y guardamos aparte la categoría I (vehículos livianos).

In [5]:
trafico = ani.pivot_table(index="mes", columns="Peaje", values="trafico", aggfunc="sum")
trafico.columns = ["trafico_guarne", "trafico_las_palmas"]
trafico["trafico_total"] = trafico["trafico_guarne"] + trafico["trafico_las_palmas"]

livianos = ani[ani["CategoriaTarifa"] == "I"].groupby("mes")["trafico"].sum()
trafico["trafico_livianos"] = livianos

trafico = trafico.reset_index()
print(trafico.shape)
trafico.head()

(149, 5)


,mes,trafico_guarne,trafico_las_palmas,trafico_total,trafico_livianos
0,2014-01-01,574186,223244,797430,581819
1,2014-02-01,455561,206812,662373,455323
2,2014-03-01,509916,233862,743778,519465
3,2014-04-01,514018,222521,736539,520308
4,2014-05-01,498184,229681,727865,498870


## 2. DANE – Encuesta Mensual de Alojamiento (EMA)
Es un Excel con muchas hojas. Usamos dos:
- **4.2** porcentaje de ocupación mensual por región,
- **6.1** índice de tarifas por tipo de acomodación (base 2019 = 100).

Nos interesa solo la columna de **Antioquia** (ojo: es Antioquia, no solo Medellín; y la tarifa es un índice, no pesos).

In [6]:
archivo = RAW + "ema_anexos_sep2026.xlsx"
ocu_raw = pd.read_excel(archivo, sheet_name="4.2 Porc Mens Ocupación.reg", header=None)
ocu_raw.iloc[10:16, :10]

,0,1,2,3,4,5,6,7,8,9
10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11,Año,Meses,Total nacional,"Bogotá, D. C.",Cartagena,Central,Caribe,Eje Cafetero,Antioquia,San Andrés y Providencia
12,2019,Enero,47.166178,44.804505,67.437514,38.745271,51.009846,49.560509,46.537566,72.012276
13,NaN,Febrero,48.248052,58.725596,65.323641,31.369122,45.726525,39.068687,45.38783,71.20798
14,NaN,Marzo,47.490489,55.613889,65.873969,31.166316,49.468632,39.481411,44.43692,70.368067
15,NaN,Abril,45.743693,53.262452,56.891085,32.934823,44.551978,40.564177,43.427767,71.010904


La tabla empieza en la fila 12 (índice). La columna 0 es el año, la 1 el mes y la columna 8 es **Antioquia**. Dos problemas:
1. El año solo aparece en enero de cada bloque → hay que **rellenarlo hacia abajo** (`ffill`).
2. Desde 2023 el año viene como `2023p` (p = provisional) → extraemos los 4 dígitos.

In [7]:
meses = {"enero": 1, "febrero": 2, "marzo": 3, "abril": 4, "mayo": 5, "junio": 6, "julio": 7,
         "agosto": 8, "septiembre": 9, "octubre": 10, "noviembre": 11, "diciembre": 12}

def leer_hoja_ema(hoja, columnas):
    d = pd.read_excel(archivo, sheet_name=hoja, header=None).iloc[12:, :].copy()
    d["anio"] = pd.to_numeric(d[0].astype(str).str.extract(r"(\d{4})")[0]).ffill()
    d["mes_num"] = d[1].astype(str).str.strip().str.lower().map(meses)
    d = d.dropna(subset=["anio", "mes_num"])            # quita notas al pie
    out = pd.DataFrame({"mes": pd.to_datetime(dict(year=d["anio"].astype(int), month=d["mes_num"].astype(int), day=1))})
    for nombre, col in columnas.items():
        out[nombre] = pd.to_numeric(d[col], errors="coerce").values
    return out

ocupacion = leer_hoja_ema("4.2 Porc Mens Ocupación.reg", {"ocupacion_ema_antioquia": 8})
print(ocupacion.shape, ocupacion["mes"].min(), ocupacion["mes"].max())
ocupacion.tail(3)

(93, 2) 2019-01-01 00:00:00 2026-09-01 00:00:00


,mes,ocupacion_ema_antioquia
102,2026-07-01,50.914198
103,2026-08-01,51.420753
104,2026-09-01,48.547428


**Tarifas.** La hoja 6.1 no trae un índice general, solo por acomodación **sencilla** (columna 38) y **doble** (columna 41) para Antioquia. Calculamos su **promedio simple** como índice resumen (decisión nuestra).

In [8]:
tarifas = leer_hoja_ema("6.1 Ind.Var Tarifas.acomoda", {"indice_tarifa_ema_sencilla": 38, "indice_tarifa_ema_doble": 41})
tarifas["indice_tarifa_ema_antioquia"] = tarifas[["indice_tarifa_ema_sencilla", "indice_tarifa_ema_doble"]].mean(axis=1)

ema = ocupacion.merge(tarifas, on="mes", how="outer")
print(ema.shape, "| duplicados:", ema["mes"].duplicated().sum())
ema.head()

(93, 5) | duplicados: 0


,mes,ocupacion_ema_antioquia,indice_tarifa_ema_sencilla,indice_tarifa_ema_doble,indice_tarifa_ema_antioquia
0,2019-01-01,46.537566,102.059113,105.041651,103.550382
1,2019-02-01,45.387830,96.163494,98.669385,97.416439
2,2019-03-01,44.436920,96.849923,99.557266,98.203594
3,2019-04-01,43.427767,98.451017,99.771105,99.111061
4,2019-05-01,44.480696,97.986075,96.493778,97.239926


## 3. Migración Colombia – Entradas de extranjeros
Trae entradas por año, mes, nacionalidad y **puesto de control migratorio (PCM)**, pero el PCM solo viene como coordenadas, no como nombre.

In [9]:
mig = pd.read_csv(RAW + "migracion_entradas_extranjeros.csv", dtype=str)
print(mig.shape)
mig["Ubicación PCM"].value_counts().head(5)

(181186, 9)


Ubicación PCM
(4.697144,-74.140516)     26473
(6.171601,-75.427454)     17634
(10.445761,-75.516429)    16612
(3.541112,-76.384584)     13877
(10.886611,-74.776708)    10772
Name: count, dtype: int64

El aeropuerto José María Córdova (Rionegro) está en las coordenadas **(6.1716, -75.4275)**; en los datos aparece con una variante casi idéntica. Filtramos esas dos.

In [10]:
jmc = ["(6.171601,-75.427454)", "(6.17134,-75.427194)"]
mig = mig[mig["Ubicación PCM"].isin(jmc)].copy()
print("Filas del aeropuerto JMC:", len(mig))

mig["Total"] = pd.to_numeric(mig["Total"].str.replace(",", ""), errors="coerce")
mig["mes_num"] = mig["Mes"].str.strip().str.lower().map(meses)
mig["mes"] = pd.to_datetime(dict(year=mig["Año"].astype(int), month=mig["mes_num"], day=1))

print("Faltantes en Total:", mig["Total"].isna().sum())

migracion = mig.groupby("mes", as_index=False)["Total"].sum()
migracion = migracion.rename(columns={"Total": "entradas_extranjeros_jmc"})
print(migracion.shape)
migracion.head()

Filas del aeropuerto JMC: 17833
Faltantes en Total: 0
(174, 2)


,mes,entradas_extranjeros_jmc
0,2012-01-01,8967
1,2012-02-01,8522
2,2012-03-01,8853
3,2012-04-01,8204
4,2012-05-01,8906


## 4. Unión en una tabla mensual
Armamos un calendario con todos los meses de **2019-01 a 2025-12** y le pegamos cada base por mes (`merge` con `how="left"`).

In [11]:
panel = pd.DataFrame({"mes": pd.date_range("2019-01-01", "2025-12-01", freq="MS")})

for base in [trafico, ema, migracion]:
    panel = panel.merge(base, on="mes", how="left")

panel.insert(1, "anio", panel["mes"].dt.year)
panel.insert(2, "mes_num", panel["mes"].dt.month)
panel["mes"] = panel["mes"].dt.strftime("%Y-%m")   # formato año-mes
panel.head()

,mes,anio,mes_num,trafico_guarne,trafico_las_palmas,trafico_total,trafico_livianos,ocupacion_ema_antioquia,indice_tarifa_ema_sencilla,indice_tarifa_ema_doble,indice_tarifa_ema_antioquia,entradas_extranjeros_jmc
0,2019-01,2019,1,749074,304798,1053872,785247,46.537566,102.059113,105.041651,103.550382,28360
1,2019-02,2019,2,593053,273802,866855,612713,45.387830,96.163494,98.669385,97.416439,29895
2,2019-03,2019,3,660017,315570,975587,703320,44.436920,96.849923,99.557266,98.203594,30289
3,2019-04,2019,4,659836,296791,956627,695277,43.427767,98.451017,99.771105,99.111061,26976
4,2019-05,2019,5,668668,318382,987050,700484,44.480696,97.986075,96.493778,97.239926,30958


## 5. Verificación de la unión
Revisamos filas por base, filas que cruzaron, meses sin dato y duplicados.

In [12]:
inicio, fin = pd.Timestamp("2019-01-01"), pd.Timestamp("2025-12-01")
bases = {"tráfico ANI": trafico, "EMA": ema, "migración": migracion}

filas = []
for nombre, b in bases.items():
    en_periodo = b[(b["mes"] >= inicio) & (b["mes"] <= fin)]
    cruzan = en_periodo["mes"].dt.strftime("%Y-%m").isin(panel["mes"]).sum()
    filas.append({"base": nombre, "meses en la base": len(b), "meses en 2019-2025": len(en_periodo), "cruzaron": cruzan})
pd.DataFrame(filas)

,base,meses en la base,meses en 2019-2025,cruzaron
0,tráfico ANI,149,84,84
1,EMA,93,84,84
2,migración,174,84,84


In [13]:
print("Filas del panel:", len(panel), "| Columnas:", panel.shape[1])
print("Meses duplicados:", panel["mes"].duplicated().sum())
print("Valores faltantes en total:", panel.isna().sum().sum())
panel.isna().sum()

Filas del panel: 84 | Columnas: 12
Meses duplicados: 0
Valores faltantes en total: 0


mes                            0
anio                           0
mes_num                        0
trafico_guarne                 0
trafico_las_palmas             0
trafico_total                  0
trafico_livianos               0
ocupacion_ema_antioquia        0
indice_tarifa_ema_sencilla     0
indice_tarifa_ema_doble        0
indice_tarifa_ema_antioquia    0
entradas_extranjeros_jmc       0
dtype: int64

## 6. Guardar

In [14]:
panel.to_csv(PROC + "panel_publico.csv", index=False)
print("Guardado:", PROC + "panel_publico.csv")

Guardado: ../data/processed/panel_publico.csv


## Limitaciones a recordar
- **EMA es de Antioquia, no de Medellín**, y su tarifa es un **índice** (base 2019 = 100), no pesos.
- Las cifras del EMA desde 2023 son **provisionales**.
- **Migración**: son entradas de extranjeros por el aeropuerto JMC (cualquier motivo de viaje), no solo turistas. El aeropuerto se identificó por coordenadas.
- **Tráfico ANI**: solo 2 peajes, con todo tipo de vehículos (no solo turistas).
- No se corrigieron valores raros ni se rellenaron meses vacíos.